In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.select import Select
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException, NoSuchWindowException, WebDriverException
from time import sleep
import pandas as pd

In [ ]:
browser = webdriver.Chrome()
wait = WebDriverWait(browser, 10)

def ensure_browser():
    global browser, wait
    try:
        if browser is None:
            raise NoSuchWindowException("Browser is not available")
        _ = browser.current_url
        if not browser.window_handles:
            raise NoSuchWindowException("Browser window is closed")
    except Exception:
        try:
            browser.quit()
        except Exception:
            pass
        browser = webdriver.Chrome()
        wait = WebDriverWait(browser, 10)
    return browser, wait

def wait_for_search_results():
    ensure_browser()
    wait.until(lambda driver: len(driver.find_elements(By.XPATH, "//div[@data-component-type='s-search-result']")) > 0)
    return browser.find_elements(By.XPATH, "//div[@data-component-type='s-search-result']")

def get_current_page_marker():
    try:
        return browser.find_element(By.CSS_SELECTOR, "span.s-pagination-item.s-pagination-selected").text.strip()
    except Exception:
        return browser.current_url

def get_next_button():
    ensure_browser()
    buttons = browser.find_elements(By.CSS_SELECTOR, "a.s-pagination-next")
    for button in buttons:
        try:
            aria_disabled = (button.get_attribute("aria-disabled") or "").lower()
            classes = button.get_attribute("class") or ""
            if button.is_displayed() and aria_disabled != "true" and "s-pagination-disabled" not in classes:
                return button
        except StaleElementReferenceException:
            continue
    return None

def open_search(keyword):
    ensure_browser()
    browser.get("https://www.amazon.eg/")
    sleep(2)
    dropdown = wait.until(EC.presence_of_element_located((By.ID, "searchDropdownBox")))
    Select(dropdown).select_by_value("search-alias=grocery")
    search_box = browser.find_element(By.ID, "twotabsearchtextbox")
    search_box.clear()
    search_box.send_keys(keyword)
    wait.until(EC.element_to_be_clickable((By.XPATH, "(//input[@type='submit'])[1]"))).click()
    wait_for_search_results()
    sleep(2)

def go_to_next_page(page_num, keyword, previous_url, previous_marker, previous_first_card):
    last_error = None
    for _ in range(3):
        try:
            next_btn = get_next_button()
            if next_btn is None:
                return False
            browser.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
            sleep(1)
            browser.execute_script("arguments[0].click();", next_btn)
            wait.until(lambda driver: driver.current_url != previous_url or get_current_page_marker() != previous_marker)
            if previous_first_card is not None:
                try:
                    wait.until(EC.staleness_of(previous_first_card))
                except TimeoutException:
                    pass
            wait_for_search_results()
            sleep(2)
            return True
        except (TimeoutException, StaleElementReferenceException, WebDriverException) as error:
            last_error = error
            sleep(2)
    if last_error is not None:
        print(f"   Next page failed on page {page_num} for {keyword}: {last_error}")
    return False

def get_top_highlights(url):
    ensure_browser()
    if not url:
        return {}
    details = {}
    try:
        browser.get(url)
        sleep(2)
        try:
            expander = browser.find_element(By.XPATH, "//a[contains(@class,'a-expander-header') and contains(.,'Top highlights')]")
            browser.execute_script("arguments[0].click();", expander)
            sleep(1)
        except Exception:
            pass

        rows = browser.find_elements(By.XPATH, "//div[contains(@class,'a-expander-content')]//tr")
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "td")
            if len(cells) >= 2:
                key = cells[0].text.strip()
                val = cells[1].text.strip()
                if key and val:
                    details[key] = val
    except Exception as e:
        print(f"Error: {e}")
    return details

def scrape_keyword(keyword, max_pages=20):
    ensure_browser()
    print(f"\nKeyword: {keyword}")
    products = []
    seen_urls = set()
    unwanted = {"Results", "Related searches", ""}

    open_search(keyword)

    page_num = 1
    while page_num <= max_pages:
        cards = wait_for_search_results()
        page_count_before = len(products)
        first_card = cards[0] if cards else None
        previous_url = browser.current_url
        previous_marker = get_current_page_marker()

        for card in cards:
            try:
                name = card.find_element(By.XPATH, ".//h2//span").text.strip()
            except Exception:
                name = ""
            if not name or name in unwanted:
                continue

            try:
                price = card.find_element(By.XPATH, ".//span[@class='a-price-whole']").text.strip()
            except Exception:
                price = ""

            try:
                url = card.find_element(By.XPATH, ".//a[@class='a-link-normal s-no-outline']").get_attribute("href")
            except Exception:
                url = ""

            if url and url in seen_urls:
                continue
            if url:
                seen_urls.add(url)

            products.append({"name": name, "price": price, "url": url, "keyword": keyword})

        page_added = len(products) - page_count_before
        print(f"   Page {page_num} | Added: {page_added} | Total: {len(products)}")

        if not go_to_next_page(page_num, keyword, previous_url, previous_marker, first_card):
            print(f"   Finished pages for {keyword}")
            break

        page_num += 1

    return products

In [ ]:
keywords = [
    'milk', 'rice', 'oil', 'tea', 'coffee',
    'juice', 'water', 'cheese', 'yogurt', 'bread',
    'chocolate', 'biscuits', 'sugar', 'flour', 'pasta',
    'soap', 'shampoo', 'detergent', 'chips', 'honey',
    'jam', 'butter', 'sauce', 'spices', 'nuts'
]

all_products = []

for kw in keywords:
    prods = scrape_keyword(kw)
    all_products.extend(prods)
    print(f"Total products until now: {len(all_products)}")

    try:
        pd.DataFrame(all_products).to_csv(
            r'C:\Users\DCS\Desktop\backup.csv',
            index=False, encoding='utf-8-sig')
        print("  Backup saved")
    except PermissionError:
        print("   File is open")

print(f"\nTotal products: {len(all_products)}\n")

for i, prod in enumerate(all_products):
    if i % 10 == 0:
        print(f" {i+1}/{len(all_products)} - {prod['name'][:50]}")

    highlights = get_top_highlights(prod["url"])
    prod.update(highlights)

    if (i+1) % 200 == 0:
        try:
            pd.DataFrame(all_products).to_csv(
                r'C:\Users\DCS\Desktop\backup_highl.csv',
                index=False, encoding='utf-8-sig')
            print("    Backup saved")
        except PermissionError:
            print("    File is open")

In [ ]:
df = pd.DataFrame(all_products)
base_cols = ["name", "price", "url", "keyword"]
detail_cols = [c for c in df.columns if c not in base_cols]
df = df[base_cols + detail_cols]

df.to_csv(
    r'C:\Users\DCS\Desktop\amazon_grocery.csv',
    index=False,
    encoding='utf-8-sig'
 )

print(f"\n saved final file\n")
print(f"  rows: {len(df)}")
print(f"  columns : {df.columns.tolist()}")
print(df.head(3).to_string())

browser.quit()